# Day 23: Deploying RAG as an API

Day 22 built a RagService that runs correctly inside a single Python process. This notebook puts a door in front of it: a small HTTP server, built entirely with the standard library, so any client (in any language) can send a request and get a response back. Every cell below starts the server on a free port and shuts it down cleanly at the end, so nothing is left hanging.

## Start the server

The server runs in a background daemon thread, on a port the OS picks (port 0), so this notebook never blocks.

In [ ]:
import sys, os, threading
sys.path.insert(0, os.getcwd())
from api_server import RagRequestHandler, build_app_state
from http.server import ThreadingHTTPServer

RagRequestHandler.app_state = build_app_state()
httpd = ThreadingHTTPServer(('127.0.0.1', 0), RagRequestHandler)
port = httpd.server_port
server_thread = threading.Thread(target=httpd.serve_forever, daemon=True)
server_thread.start()
base_url = f'http://127.0.0.1:{port}'
print('Server started on', base_url)


## Send requests

Real HTTP requests over urllib.request: a valid query, an empty query, and a health check.

In [ ]:
import json, urllib.request, urllib.error

def send(method, path, body=None):
    data = json.dumps(body).encode() if body is not None else None
    req = urllib.request.Request(f'{base_url}{path}', data=data, method=method)
    try:
        with urllib.request.urlopen(req, timeout=5) as resp:
            return resp.status, json.loads(resp.read())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read())

status, body = send('POST', '/query', {'query': 'What is RAG?'})
print('valid query:', status, body['status'], body['sources'])

status, body = send('POST', '/query', {'query': ''})
print('empty query:', status, body['status'])

status, body = send('GET', '/health')
print('health:', status, body)


## Client example

The copyable ask() function, with timeouts and readable errors instead of raw exceptions.

In [ ]:
from client_example import ask
result = ask('What is a vector database?', base_url)
print(result['status'])
print(result['answer'][:150])
print(result['sources'])


## Status code mapping

How Day 22's service statuses map to HTTP codes: a contract any client can rely on.

In [ ]:
from request_handling_rules import STATUS_TO_HTTP_CODE, http_code_for_status
for s, c in STATUS_TO_HTTP_CODE.items():
    print(s, '->', c)


## Auth and rate limit

A request with no API key (401), a valid key (200), and a burst that exceeds the fixed-window rate limit (429).

In [ ]:
import threading as _threading
from http.server import ThreadingHTTPServer as _THS
from auth_and_rate_limit import ProtectedRagRequestHandler, get_configured_api_key
from api_server import build_app_state as _build_app_state

ProtectedRagRequestHandler.app_state = _build_app_state()
protected_httpd = _THS(('127.0.0.1', 0), ProtectedRagRequestHandler)
protected_port = protected_httpd.server_port
_threading.Thread(target=protected_httpd.serve_forever, daemon=True).start()

# A fresh http.client connection per call, with a small retry on a pure
# connection-level failure: rapid back-to-back new connections against a
# threaded server can occasionally be torn down by the OS, unrelated to
# this server's own logic, so a brief retry is standard defensive client
# behavior rather than a workaround for a real bug.
import http.client

def send_protected(headers=None):
    all_headers = {**(headers or {}), 'Content-Type': 'application/json'}
    data = json.dumps({'query': 'What is RAG?'}).encode()
    last_error = None
    for _ in range(3):
        conn = http.client.HTTPConnection('127.0.0.1', protected_port, timeout=5)
        try:
            conn.request('POST', '/query', body=data, headers=all_headers)
            resp = conn.getresponse()
            resp.read()
            return resp.status
        except (ConnectionAbortedError, ConnectionResetError, OSError) as e:
            last_error = e
        finally:
            conn.close()
    raise last_error

print('no key:', send_protected())
print('valid key:', send_protected({'X-API-Key': get_configured_api_key()}))
burst = [send_protected({'X-API-Key': get_configured_api_key()}) for _ in range(15)]
print('burst statuses:', burst)

protected_httpd.shutdown()
protected_httpd.server_close()


## HTTP vs in-process latency

The exact same RagService work, measured over a real socket versus a direct Python method call.

In [ ]:
from load_test_http import run_http_load_test
import sys as _sys, os as _os
_sys.path.insert(0, _os.path.join(_os.getcwd(), '..', 'day-22-production-rag'))
import load_test as day22_load_test
from config import Config
from rag_service import RagService

http_results = run_http_load_test(base_url, num_requests=50)
print('HTTP p50/p95 (ms):', round(http_results['p50_ms'], 4), round(http_results['p95_ms'], 4))

service = RagService(Config())
in_process_results = day22_load_test.run_load_test(service, num_requests=50)
print('in-process p50/p95 (ms):', round(in_process_results['p50_ms'], 4), round(in_process_results['p95_ms'], 4))


## Key takeaways

- An API is a fixed-rule door in front of a service: a client never needs to know how the kitchen behind it actually works.
- The same RagResponse statuses Day 22 built map to specific HTTP codes, a contract clients can rely on without it drifting.
- Oversized or malformed request bodies must close the connection, not just reject the request, or a later request on the same connection can be misread.
- Timeouts and readable error handling belong on the client side too, not just the server.
- Over a real network, latency is measurably higher than an in-process call, the exact same underlying work still costs more once a socket is involved.
- Keys belong in environment variables, never in source code, and a rate limit protects the service from any single caller overwhelming it.

In [ ]:
httpd.shutdown()
httpd.server_close()
print('Server shut down cleanly.')
